# Minimum Temperature (tasmin) Climate Indices

This notebook computes standard ETCCDI / ECA&D climate indices from a **daily** `tasmin`
NetCDF file using [`icclim`](https://icclim.readthedocs.io/).

**Assumptions (edit the Configuration cell below if any of these don't match your data):**
- Single `.nc` file for `tasmin` covering the whole period
- Units already in `degC`
- Baseline period for percentile-based indices: **1981–2010**
- Output aggregation: annual (`slice_mode="year"`) — change to `"month"`, `"DJF"`, `"JJA"`, etc. if needed

Covers warmest/coldest-night extremes, warm/cool night percentiles, tropical nights (TR), frost days (FD/CFD), and the Cold Spell Duration Index (CSDI).

Install requirements once (uncomment if needed):


In [ ]:
# !pip install icclim xarray netCDF4 matplotlib pandas --quiet

In [ ]:
import os
import warnings
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import icclim

warnings.filterwarnings("ignore")


In [ ]:
# ============================================================
# CONFIGURATION — EDIT THESE PATHS/SETTINGS FOR YOUR DATA
# ============================================================

IN_FILE = "/path/to/your/tasmin_daily.nc"          # path to your daily tasmin .nc file
VAR_NAME = "tasmin"                  # variable name inside the NetCDF file
OUT_DIR = "./outputs/tasmin"     # where results will be saved
UNIT = "degC"                         # expected units of the variable

TIME_RANGE = None                       # e.g. ("1980-01-01", "2020-12-31"); None = use full file
BASE_PERIOD_TIME_RANGE = ('1981-01-01', '2010-12-31')  # reference period for percentile indices
SLICE_MODE = "year"                     # "year", "month", "DJF", "MAM", "JJA", "SON", ...

os.makedirs(OUT_DIR, exist_ok=True)


## 1. Load & sanity-check the data

In [ ]:
# Quick look at the data + sanity-check the units attribute
ds = xr.open_dataset(IN_FILE)
print(ds)

da = ds[VAR_NAME]
print("\nDeclared units in file:", da.attrs.get("units", "MISSING"))

# icclim/xclim need a correct CF 'units' attribute to interpret the data.
# If it's missing or wrong, set it explicitly here (uncomment + fix as needed):
# ds[VAR_NAME].attrs["units"] = UNIT

ds.close()


## 2. Compute the indices

In [ ]:
# ============================================================
# INDICES TO COMPUTE
# ============================================================
# Percentile-based indices (need BASE_PERIOD_TIME_RANGE): tn10p, tn90p, csdi

INDICES = ["TNx", "TNn", "TN10p", "TN90p", "TR", "FD", "CFD", "CSDI"]

results = {}

for idx_name in INDICES:
    print(f"Computing {idx_name} ...")
    out_file = os.path.join(OUT_DIR, f"{idx_name}_tasmin.nc")
    try:
        needs_base_period = idx_name.lower() in ['tn10p', 'tn90p', 'csdi']
        ds_out = icclim.index(
            index_name=idx_name,
            in_files=IN_FILE,
            var_name=VAR_NAME,
            slice_mode=SLICE_MODE,
            time_range=TIME_RANGE,
            base_period_time_range=BASE_PERIOD_TIME_RANGE if needs_base_period else None,
            out_file=out_file,
        )
        results[idx_name] = ds_out
        print(f"  -> saved to {out_file}")
    except Exception as e:
        print(f"  !! FAILED for {idx_name}: {e}")


## 3. Quick-look plots + CSV summary

In [ ]:
# ============================================================
# QUICK-LOOK PLOTS (spatial mean time series) + CSV EXPORT
# ============================================================
summary = {}

for idx_name, ds_out in results.items():
    data_vars = [v for v in ds_out.data_vars if not v.endswith("_thresholds")]
    if not data_vars:
        continue
    da = ds_out[data_vars[0]]

    spatial_dims = [d for d in da.dims if d not in ("time",)]
    ts = da.mean(dim=spatial_dims, skipna=True) if spatial_dims else da

    summary[idx_name] = ts.to_series()

    plt.figure(figsize=(7, 3))
    ts.plot()
    plt.title(f"{idx_name} — spatial mean")
    plt.xlabel("Time")
    plt.ylabel(idx_name)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{idx_name}_tasmin_timeseries.png"), dpi=120)
    plt.show()

if summary:
    df_summary = pd.DataFrame(summary)
    csv_path = os.path.join(OUT_DIR, "tasmin_indices_summary.csv")
    df_summary.to_csv(csv_path)
    print(f"Summary CSV saved to: {csv_path}")
    df_summary


## 4. Output files

In [ ]:
# List everything that was produced
for f in sorted(os.listdir(OUT_DIR)):
    print(f)
